# Analyze forces, frequency spectrum and convergence 

In [ ]:
import matplotlib.pyplot as plt

from os import makedirs
from os.path import join, exists
from torch import where, tensor, argmax, from_numpy, pi

from utils import load_yplus, load_ratio_rans_les, load_force_coeffs, interpolate_uniform, compute_fft, compute_norm_of_fields, load_residuals

In [ ]:
# flow quantities
# u_inf = 238.845

# validation @ Ma = 0.73
u_inf = 242.16629

# sweep
# u_inf = 275.795 

# chord length
chord = 1

# start of quasi-steady state -> chosen based on course of cl
tstar_start = 145
tstar_end = 250

# use latex fonts
plt.rcParams.update({"text.usetex": True, "figure.dpi": 360, "text.latex.preamble": r"\usepackage{xcolor}"})

# use default color cycle
color = [f"C{i}" for i in range(10)]
ls = ["-"] * 10

In [ ]:
# define load and save path
# validation
load_dir = join("/media", "janis", "Elements", "Janis", "2D_buffet_simulation")
save_dir = join("..", "run", "plots", "DDES_URANS_validation_comparison_alpha_3.5deg")
cases = ["URANS_2D_Ma0.73_Re3e6/URANS_SA_alpha3.5deg_blockMesh", "URANS_2D_Ma0.73_Re3e6/URANS_SALSA_alpha3.5deg_blockMesh_useRmod_useSmod",
         "DDES_3D_Ma0.73_Re3e6/DDES_SA_Re3e6_Ma0.73_alpha3.5deg_y65_ymax0.25"]
legend = [r"$\mathrm{URANS}~(\mathrm{SA})$, $N_{\mathrm{cells}, y} = 1$", r"$\mathrm{URANS}~(\mathrm{SALSA})$, $N_{\mathrm{cells}, y} = 1$",
          r"$\mathrm{DDES}~(\mathrm{SA})$, $N_{\mathrm{cells}, y} = 65$"]

In [ ]:
load_dir = join("/media", "janis", "Elements", "Janis", "2D_buffet_simulation", "URANS_2D_Ma0.73_Re3e6")
save_dir = join("..", "run", "plots", "URANS_validation", "URANS_blockMesh", "comparison_all_turbulence_models_newMesh", "3.5deg")
cases = ["URANS_SA_alpha3.5deg_blockMesh_newMesh", "URANS_kOmegaSST_alpha3.5deg_blockMesh_newMesh",
         "URANS_SALSA_alpha3.5deg_blockMesh_useRmod_useSmod_newMesh", "URANS_SA_rc_alpha3.5deg_blockMesh_newMesh",
         "URANS_RSM_SSG_alpha3.5deg_blockMesh_newMesh_couplingIter_0", "URANS_RSM_LRR_alpha3.5deg_blockMesh_newMesh_couplingIter_0"]

legend = [r"$\mathrm{SA}$", r"$\mathrm{k-}\omega \mathrm{~SST}$", r"$\mathrm{SALSA}$", r"$\mathrm{SA-rc}$",
          r"$\mathrm{SSG}~\mathrm{(RSM)}$", r"$\mathrm{LRR}~\mathrm{(RSM)}$"]

In [ ]:
# create plot directory
if not exists(save_dir):
    makedirs(save_dir)

In [ ]:
# load cl and cd
forces = [load_force_coeffs(join(load_dir, c)) for c in cases]

In [ ]:
# plot only cl
fig, ax = plt.subplots(figsize=(6, 3))
for i in range(len(cases)):
    ax.plot(forces[i].t * u_inf / chord, forces[i].cy, label=legend[i], ls=ls[i], color=color[i])

    # print mean cl
    """try:
        idx_start = where(abs(tensor(forces[i]["t"].values) * u_inf / chord - tstar_start) < 1e-3)[0][0].item()
    except IndexError:
        print(f"Mean cl case {i}:\t{round(forces[i].cy.mean(), 3)}")
    try:
        idx_end = where(abs(tensor(forces[i]["t"].values) * u_inf / chord - tstar_end) < 1e-3)[0][0].item()
    except IndexError:
        print(f"Mean cl case {i}:\t{round(forces[i].cy[idx_start:].mean(), 3)}")
    else:
        print(f"Mean cl case {i}:\t{round(forces[i].cy[idx_start:idx_end+1].mean(), 3)}")"""

    # mark mean
    # ax.axhline(forces[i].cy[idx_start:].mean(), ls="-.", color=color[i], lw=1.25)

ax.set_ylabel(r"$c_l$")
ax.set_xlabel(r"$t \frac{U_{\infty}}{c}$")
ax.set_xlim(0, forces[-1].t.iloc[-1] * u_inf / chord)
# ax.set_xlim(0, 50)
ax.set_ylim(0.75, 1.02)

fig.tight_layout()
ax.grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")

# mark beginning of investigated time window
ax.axvline(tstar_start, ls=":", color="red")

# mark min. & max. values for cl of Johannes paper
# ax.axhspan(0.8, 1.1, 0, 1, color="black", alpha=0.3, label=r"$\mathrm{Kleinert~et~al.}$", zorder=0)

# mark min. & max. and mean values for cl of Giannelis et al. (Ma = 0.73, Re = 3e6, alpha = 3.5deg), approx.
ax.axhspan(0.814, 0.965, 0, 1, color="black", alpha=0.3, label=r"$\mathrm{Giannelis~et~al.}$", zorder=0)
# ax.axhline(0.89, ls="-.", color="black", zorder=1, lw=1, label=r"$\bar{c}_L$")

fig.legend(ncol=3, loc="upper center")
fig.subplots_adjust(top=0.86)
plt.savefig(join(save_dir, "comparison_cl_vs_t.png"))
plt.show()

In [ ]:
# plot max y+ vs. time
y_plus = [load_yplus(join(load_dir, c)) for c in cases]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
for i in range(len(cases)):
    ax.plot(y_plus[i].t * u_inf / chord, y_plus[i].yPlus_max, label=legend[i], ls=ls[i])

ax.set_xlabel(r"$t \frac{U_{\infty}}{c}$")
ax.set_ylabel("$y^+_{max}$")
ax.set_xlim(y_plus[0].t.iloc[0] * u_inf / chord, y_plus[0].t.iloc[-1] * u_inf / chord)
ax.set_ylim(0.9, 1.4)

# mark beginning of investigated time window
# ax.axvline(tstar_start, ls=":", color="red")
fig.tight_layout()
ax.grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")
fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.8)
plt.savefig(join(save_dir, "comparison_yPlus_max_vs_t.png"))
plt.show()

In [ ]:
# plot avg. y+ vs. time
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
for i in range(len(cases)):
    ax.plot(y_plus[i].t * u_inf / chord, y_plus[i].yPlus_avg, label=legend[i], ls=ls[i])
        
ax.set_xlabel(r"$t \frac{U_{\infty}}{c}$")
ax.set_ylabel("$y^+_{avg}$")
ax.set_xlim(y_plus[0].t.iloc[0] * u_inf / chord, y_plus[0].t.iloc[-1] * u_inf / chord)
ax.set_ylim(0.3, 0.8)

# mark beginning of investigated time window
# ax.axvline(tstar_start, ls=":", color="red", zorder=0)
fig.tight_layout()
ax.grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")
fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.8)
plt.savefig(join(save_dir, "comparison_yPlus_avg_vs_t.png"))
plt.show()

In [ ]:
# re-sample the force coefficients since the simulation is run with a variable time step
t_uniform, cy_inter = [], []
for f in forces:
    try:
        idx_start = where(abs(tensor(f["t"].values) * u_inf / chord - tstar_start) < 1e-4)[0][0].item()

        # remove the transient phase and duplicates (resulting from dt < write precision)
        f.drop(list(range(0, idx_start)), inplace=True)
        f.reset_index(inplace=True, drop=True)
        idx_end = where(abs(tensor(f["t"].values) * u_inf / chord - tstar_end) < 1e-4)[0][0].item()
        f.drop(list(range(idx_end, f.index[-1]+1)), inplace=True)
    except IndexError:
        pass
        
    f.reset_index(inplace=True, drop=True)

    # interpolate values
    _t_uniform, _cy_inter = interpolate_uniform(f["t"].values, f["cy"].values)
    t_uniform.append(_t_uniform)
    cy_inter.append(_cy_inter)

In [ ]:
# verify interpolation for all cases
for i, c in enumerate(cases):
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(forces[i]["t"] * u_inf / chord, forces[i]["cy"], label=r"$\mathrm{original}$")
    ax.plot(t_uniform[i] * u_inf / chord, cy_inter[i], ls="--", label=r"$\mathrm{interpolated}$")
    ax.set_xlabel(r"$t \frac{U_{\infty}}{c}$")
    ax.set_ylabel(r"$c_l$")
    ax.set_xlim(t_uniform[i][0] * u_inf / chord, t_uniform[i][-1] * u_inf / chord)
    ax.grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
    ax.minorticks_on()
    ax.tick_params(axis="x", which="minor", bottom=False)
    ax.grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")
    fig.tight_layout()
    fig.legend(ncols=2, loc="upper center")
    fig.subplots_adjust(top=0.88)
    plt.savefig(join(save_dir, f"check_interpolation_{c.split("/")[-1]}.png"))
    plt.show()

In [ ]:
# compute FFT
results_fft_cl = [compute_fft(cy_inter[c], (t_uniform[c][1] - t_uniform[c][0]).item()) for c in range(len(cases))]

idx = [argmax(from_numpy(res[1])) for res in results_fft_cl]

# print out buffet frequencies for each case
for i, res in enumerate(results_fft_cl):
    print(f"Max. amplitude at f = {round(res[0][idx[i]].item(), 5)} Hz (Sr = {round(res[0][idx[i]].item() * chord / u_inf, 5)})")

In [ ]:
# plot frequency of max. amplitude as Sr and Hz
f_legend, sr_legend = [], []

fig, ax = plt.subplots(2, 1, figsize=(6, 5), sharey="col")
for i, fa in enumerate(results_fft_cl):
    ax[0].plot(fa[0] * 2 * pi * chord/u_inf, fa[1], label=legend[i])
    ax[1].plot(fa[0] * chord / u_inf, fa[1])
    
    # not working -> remains black
    f_legend.append(r"$\textcolor{" + color[i] + "}{f = " + "{:.2f}".format(fa[0][idx[i]].item()) + r"\, Hz" + "}$")
    sr_legend.append(r"$\textcolor{" + color[i] + "}{Sr = " + "{:.4f}".format(fa[0][idx[i]].item() * chord / u_inf) + "}$")

# ax[0].set_xscale("log")
ax[0].set_xlim(0, 1)
ax[1].set_xlim(0.02, 0.14)
ax[0].set_xlabel(r"$\frac{2\pi f c}{U_\infty}$ $[-]$")
ax[1].set_xlabel("$Sr$ $[-]$")

for i in range(2):
    ax[i].set_ylabel(r"$\mathrm{PSD}$")
    ax[i].ticklabel_format(style="sci", axis="y", scilimits=(0, 0))

# mark Sr of literature TODO: check values for Jacquin et al. -> replace legend
ax[0].axvspan(0.056 * 2 * pi, 0.076 * 2 * pi, 0, 1, color="black", alpha=0.3, label=r"$\mathrm{Kleinert~et~al.}$", zorder=0)
ax[1].axvspan(0.056, 0.076, 0, 1, color="black", alpha=0.3, zorder=0)

fig.tight_layout()
fig.legend(ncols=3, loc="upper center")
fig.subplots_adjust(top=0.9)
plt.savefig(join(save_dir, "dominant_frequency.png"))
plt.show()

In [ ]:
# plot complete FFT
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
for i, res in enumerate(results_fft_cl):
    ax.plot(res[0] * chord / u_inf, res[1], label=legend[i])
ax.set_yscale("log")
ax.set_xscale("log")
ax.set_xlabel(r"$Sr$ $[\mathrm{-}]$")
ax.set_ylabel(r"$\mathrm{PSD}$")
ax.set_ylim(1e-10, 1e-2)
ax.set_xlim(1e-2, 1e1)
fig.tight_layout()
ax.grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="x")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="x")
fig.legend(ncol=4, loc="upper center")
fig.subplots_adjust(top=0.86)
plt.savefig(join(save_dir, "results_fft.png"))
plt.show()

In [ ]:
# load residuals
residuals = [load_residuals(join(load_dir, c)) for c in cases]

# plot residuals
for c in range(len(cases)):
    fig, ax = plt.subplots(2, 1, figsize=(6, 5), sharex="col")
    for k in residuals[c].keys():
        if k.endswith("_initial") and not k == "Time":
            ax[0].plot(residuals[c]["Time"] * u_inf / chord, residuals[c][k], label=k)
    
        elif k.endswith("_final") and not k == "Time":
            ax[1].plot(residuals[c]["Time"] * u_inf / chord, residuals[c][k], label=k)
    
    for i in range(2):
        # ax[i].axvline(750, color="black", ls="--")
    
        ax[i].set_yscale("log")
        ax[i].set_xscale("log")
        ax[i].set_xlim(residuals[c]["Time"].min() * u_inf / chord, residuals[c]["Time"].max() * u_inf / chord)
        ax[i].legend(ncols=2, loc="lower left", facecolor="white", framealpha=1)
    
        ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
        ax[i].minorticks_on()
        ax[i].tick_params(axis="x", which="minor", bottom=False)
        ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")
    
    # ax[-1].set_xlabel(r"iteration no. $\#$")
    ax[-1].set_xlabel(r"$t \frac{U_{\infty}}{c}$")
    fig.tight_layout()
    plt.savefig(join(save_dir, f"residuals_vs_iterations_case{c}.png"))

In [ ]:
# compute the convergence of UMean wrt time
results = [compute_norm_of_fields(join(load_dir, c)) for c in cases]

fig, ax = plt.subplots(1, 1, figsize=(6, 3))
i = 0
for times, res in results:
    ax.plot(times * u_inf / chord, res, marker="x", label=legend[i])
    i += 1
ax.set_xlim(results[0][0][0] * u_inf / chord, results[0][0][-1] * u_inf / chord)
ax.set_yscale("log")
ax.set_xlabel(r"$t \frac{U_{\infty}}{d}$")
ax.set_ylabel(r"$\left| \left| \frac{\overline{U}_{t+1} - \overline{U}_{t}}{\Delta t}\right| \right|_F \times "
              r"\frac{1}{||\overline{U}_{t_0} ||_F}$")
ax.grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")
fig.tight_layout()
fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.82)
plt.savefig(join(save_dir, f"convergence_Umean_field_vs_time.png"))
plt.show()

In [ ]:
# plot ratio RANS / LES vs. time
ratio = [load_ratio_rans_les(join(load_dir, c)) for c in cases]

ls = ["-"] * len(ratio)
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
for i in range(len(ratio)):
    ax.plot(ratio[i].t * u_inf / chord, ratio[i].rans / ratio[i].les, label=legend[i], ls=ls[i])
        
ax.set_xlabel(r"$t \frac{U_{\infty}}{c}$")
ax.set_ylabel("$RANS / LES$")
ax.set_yscale("log")

# mark beginning of investigated time window
ax.axvline(tstar_start, ls=":", color="red")
# ax.set_xlim(ratio[0].t.iloc[0] * u_inf / chord, ratio[0].t.iloc[-1] * u_inf / chord)
ax.set_xlim(0, tstar_end)
fig.tight_layout()
ax.grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")
fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.81)
plt.savefig(join(save_dir, "comparison_RANS_LES.png"))
plt.show()

In [ ]:
# check forces 
forces = [load_force_coeffs(join(load_dir, c)) for c in cases]

ls = ["-"] * len(forces)

# use default color cycle
color = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f"]

fig, ax = plt.subplots(2, 1, figsize=(6, 4), sharex="col")
for i in range(len(cases)):
    ax[0].plot(forces[i].t * u_inf / chord, forces[i].cx, label=legend[i], ls=ls[i], color=color[i])
    ax[1].plot(forces[i].t * u_inf / chord, forces[i].cy, ls=ls[i], color=color[i])

    # print mean cl
    idx_start = where(abs(tensor(forces[i]["t"].values) * u_inf / chord - tstar_start) < 1e-3)[0][0].item()
    
    try:
        idx_end = where(abs(tensor(forces[i]["t"].values) * u_inf / chord - tstar_end) < 1e-3)[0][0].item()
    except:
        print(f"Mean cl case {i}:\t{round(forces[i].cy[idx_start:].mean(), 3)}")
    else:
        print(f"Mean cl case {i}:\t{round(forces[i].cy[idx_start:idx_end+1].mean(), 3)}")

    # mark mean
    ax[1].axhline(forces[i].cy[idx_start:].mean(), ls="-.", color=color[i], lw=1.25)

ax[0].set_ylim(0, 0.02)
#ax[1].set_ylim(0.5, 1.2)
ax[0].set_ylabel(r"$c_d$")
ax[1].set_ylabel(r"$c_l$")
ax[-1].set_xlabel(r"$t \frac{U_{\infty}}{c}$")
ax[-1].set_xlim(0, forces[0].t.iloc[-1] * u_inf / chord)

fig.tight_layout()
for i in range(2):
    ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
    ax[i].minorticks_on()
    ax[i].tick_params(axis="x", which="minor", bottom=False)
    ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")
    
    # mark beginning of investigated time window
    # ax[i].axvline(161.44, ls=":", color="black", zorder=1, lw=1)
    # ax[i].axvline(93.6, ls=":", color="black", zorder=1, lw=1)
    # ax[i].axvline(tstar_start, ls=":", color="red")

# mark min. & max. values for cl of Johannes paper
# ax[1].axhspan(0.8, 1.1, 0, 1, color="black", alpha=0.3, label="$literature$", zorder=0)

# mark min. & max. and mean values for cl of Giannelis et al (Ma = 0.73, Re = 3e6, alpha = 3.5deg), approx.
# ax[1].axhspan(0.81, 0.97, 0, 1, color="black", alpha=0.3, label="Giannelis et al", zorder=0)
# ax[1].axhline(0.89, ls="-.", color="black", zorder=1, lw=1, label=r"$\bar{c}_L$")

fig.legend(ncol=3, loc="upper center")
fig.subplots_adjust(top=0.9)
plt.savefig(join(save_dir, "comparison_coefficients_vs_t.png"))
plt.show()